# CIDS · 01 — S0: SAM-Med3D embeddings + zero-shot baseline

**Flow:** GitHub `Vemuri-BK/cids` → clone here → run.  
**Settings:** Accelerator = GPU T4 x2 (one GPU is used), Internet = ON (downloads SAM-Med3D-turbo from Hugging Face).  
**Input:** attach the 2-channel MAMA-MIA cache (`mama-mia-preprocessed-cache-2ch`). If this account does not own it, share it from the owner account first (dataset → Settings → Sharing).

What it does for all 1506 cases: 1 mm → 1.5 mm, 128³ crops (train: 1 centred + 3 jittered; test: 1 centred), two SAM inputs (post1−pre subtraction, post1), frozen SAM-Med3D encoder → 384×8³ embeddings. Test cases also get the zero-shot click baseline (1–5 clicks). Resumable.

**After it finishes:** Output → *New Dataset* → name it `cids-sam-emb` (≈3–4 GB).

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'   # public repo, no token needed
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = ''               # '' = auto-detect the folder that contains train/ and val/
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/sam_emb'

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q medim einops appdirs
!pytest -q tests

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart the session.'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i); print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
import sys; sys.path.insert(0, CODE_DIR)
from cids.data.cache import find_cache_root, list_cases
root = find_cache_root(CACHE_ROOT); cases = list_cases(root)
print('cache:', root, '|', sum(c[1]=='train' for c in cases), 'train /', sum(c[1]=='test' for c in cases), 'test')
assert len(cases) == 1506, 'expected 1506 cases'
CACHE_ROOT = str(root)

## Smoke test — 3 train + 3 test cases (downloads the checkpoint, ~2 min)
Check: no errors, and zero-shot 1-click Dice clearly above 0 (a near-zero value means the checkpoint or the input normalisation is wrong — stop and send me the output).

In [ ]:
!python scripts/s0_sam_embeddings.py --cache_root {CACHE_ROOT} --out /kaggle/working/s0_smoke --limit 3
!rm -rf /kaggle/working/s0_smoke   # keep it out of the final output

## Full run — all 1506 cases (~1–2 h)

In [ ]:
!python scripts/s0_sam_embeddings.py --cache_root {CACHE_ROOT} --out {OUT}

In [ ]:
import pandas as pd
zs = pd.read_csv(f'{OUT}/zero_shot_clicks.csv').drop_duplicates(['cache_id','mode','clicks'], keep='last')
ix = pd.read_csv(f'{OUT}/index.csv').drop_duplicates('cache_id', keep='last')
print('cases indexed:', len(ix))
print(zs.pivot_table(index='clicks', columns='mode', values='dice', aggfunc=['mean','median']).round(4))
print(ix[ix.tumor_frac_in_crop0 < 0.95].sort_values('tumor_frac_in_crop0').head(10))   # tumours cut by the crop
!du -sh {OUT}